In [ ]:
from typing import Tuple
import pandas as pd
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from pytorch_lightning import LightningModule, Trainer
from torch_geometric.nn import GCNConv
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


from sklearn.metrics import precision_score, recall_score, f1_score

import os, sys
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from dotenv import load_dotenv
load_dotenv(dotenv_path="../.env")
MLFLOW_SERVICE_URI = os.getenv("MLFLOW_SERVICE_URI", "")

MOVIELENS_DATA_DIR = "../datasets/movielens-2k/user_ratedmovies.dat"

In [8]:
interaction_df = pd.read_table(os.path.join(MOVIELENS_DATA_DIR))
print("data count:", len(interaction_df))
interaction_df.head()

data count: 855598


,userID,movieID,rating,date_day,date_month,date_year,date_hour,date_minute,date_second
0,75,3,1.0,29,10,2006,23,17,16
1,75,32,4.5,29,10,2006,23,23,44
2,75,110,4.0,29,10,2006,23,30,8
3,75,160,2.0,29,10,2006,23,16,52
4,75,163,4.0,29,10,2006,23,29,30


In [34]:
interaction_df = interaction_df.loc[interaction_df["date_year"].between(2006, 2008), :].reset_index(drop=True)
print("data count:", len(interaction_df))
NUM_USER = interaction_df["userID"].nunique()
NUM_ITEM = interaction_df["movieID"].nunique()
print("num of distinct users:", NUM_USER)
print("num of distinct items:", NUM_ITEM)

data count: 480608
num of distinct users: 2103
num of distinct items: 9519


In [10]:
from collections import Counter

# TODO: change this to filter out cold-start user/item
MIN_USER_NUM, MIN_ITEM_NUM = 1, 1
USER_ID_FIELD = "userID"
ITEM_ID_FIELD = "movieID"


def get_illegal_ids_by_inter_num(
    df: pd.DataFrame,
    field: str,
    max_num: int = None,
    min_num: int = None,
    verbose: bool = False,
) -> set:
    if field is None:
        return set()
    if max_num is None and min_num is None:
        return set()

    max_num = max_num or np.inf
    min_num = min_num or -1

    ids = df[field].values
    inter_num = Counter(ids)
    ids = {id_ for id_ in inter_num if inter_num[id_] < min_num or inter_num[id_] > max_num}
    if verbose:
        print(f"{len(ids)} illegal_ids_by_inter_num, field={field}")

    return ids


def filter_by_k_core(
        raw_df: pd.DataFrame,
        verbose: bool = False,
    ) -> pd.DataFrame:
    df = raw_df.copy()
    while True:
        ban_users = get_illegal_ids_by_inter_num(df, field=USER_ID_FIELD, max_num=None, min_num=MIN_USER_NUM, verbose=verbose)
        ban_items = get_illegal_ids_by_inter_num(df, field=ITEM_ID_FIELD, max_num=None, min_num=MIN_ITEM_NUM, verbose=verbose)
        if len(ban_users) == 0 and len(ban_items) == 0:
            print("done!")
            return df.reset_index(drop=True)

        dropped_inter = pd.Series(False, index=df.index)
        if USER_ID_FIELD:
            dropped_inter |= df[USER_ID_FIELD].isin(ban_users)
        if ITEM_ID_FIELD:
            dropped_inter |= df[ITEM_ID_FIELD].isin(ban_items)
        if verbose:
            print(f"{len(dropped_inter)} dropped interactions")
        df.drop(df.index[dropped_inter], inplace=True)


In [11]:
df = filter_by_k_core(interaction_df, verbose=True)
print("Data count before filtering:", len(interaction_df))
print("Data count after filtering:", len(df))


0 illegal_ids_by_inter_num, field=userID
0 illegal_ids_by_inter_num, field=movieID
done!
Data count before filtering: 480608
Data count after filtering: 480608


### Re-index
NOTE: if NO filtering, the re-index has no impact on the dataset

In [12]:
u_mapping_file = 'userid_mapping.csv'
i_mapping_file = 'itemid_mapping.csv'
uid_field, iid_field = "userID", "movieID"

uni_users = sorted(pd.unique(df[uid_field]))
uni_items = sorted(pd.unique(df[iid_field]))

# start from 0
u_id_map = {k: i for i, k in enumerate(uni_users)}
i_id_map = {k: i for i, k in enumerate(uni_items)}

df[uid_field] = df[uid_field].map(u_id_map)
df[iid_field] = df[iid_field].map(i_id_map)
df[uid_field] = df[uid_field].astype(int)
df[iid_field] = df[iid_field].astype(int)

# dump
rslt_dir = './'
u_df = pd.DataFrame(list(u_id_map.items()), columns=['from', 'to'])
i_df = pd.DataFrame(list(i_id_map.items()), columns=['from', 'to'])

u_df.to_csv(os.path.join(rslt_dir, u_mapping_file), index=False)
i_df.to_csv(os.path.join(rslt_dir, i_mapping_file), index=False)
print(f'mapping dumped...')

mapping dumped...


In [13]:
# NOTE: check out the re-index mapping
uid_mapping_df = pd.read_csv(u_mapping_file)
uid_mapping_df

# iid_mapping_df = pd.read_csv(i_mapping_file)
# iid_mapping_df

,from,to
0,75,0
1,78,1
2,127,2
3,170,3
4,175,4
...,...,...
2098,71497,2098
2099,71509,2099
2100,71525,2100
2101,71529,2101


### Building Bi-partite Graph

In [14]:
# NOTE: defining the postive samples with threshold = 4.0
THRESHOLD = 4.0
df['label'] = (df['rating'] >= THRESHOLD).astype(int)

train_df = df.loc[df["date_year"].between(2006, 2007), :].reset_index(drop=True)
valid_df = df.loc[(df["date_year"] == 2008) & (df["date_month"].between(1, 6)), :].reset_index(drop=True)
test_df = df.loc[(df["date_year"] == 2008) & (df["date_month"].between(7, 12)), :].reset_index(drop=True)

total_cnt = len(df)
print("Splitting data into train/valid/test by time period:")
print(f"train: {len(train_df)} ({round(len(train_df) / total_cnt * 100, 2)}%)")
print(f"valid: {len(valid_df)} ({round(len(valid_df) / total_cnt * 100, 2)}%)")
print(f"test: {len(test_df)} ({round(len(test_df) / total_cnt * 100, 2)}%)")
print("---"*10, "\n")

print("Check target label distribution after splitting:")
print("train", train_df["label"].value_counts(), "\n", end="")
print("valid", valid_df["label"].value_counts(), "\n", end="")
print("test", test_df["label"].value_counts(), "\n", end="")



Splitting data into train/valid/test by time period:
train: 340207 (70.79%)
valid: 64761 (13.47%)
test: 75640 (15.74%)
------------------------------ 

Check target label distribution after splitting:
train label
0    194079
1    146128
Name: count, dtype: int64 
valid label
0    36444
1    28317
Name: count, dtype: int64 
test label
0    41484
1    34156
Name: count, dtype: int64 


In [24]:
from torch_geometric.data import Data

def create_interaction_graph(df_split: pd.DataFrame) -> Data:
    # NOTE: drop negative samples to construct graph of positive connectivity
    df_split = df_split.loc[df_split["label"] == 1, :]

    edge_index = torch.tensor([df_split['userID'].values, df_split['movieID'].values], dtype=torch.long)
    edge_index = torch.cat((edge_index, edge_index[[1, 0]]), dim=1) # [2, num_edges*2]
    edge_label = torch.tensor(df_split['label'].values, dtype=torch.float)
    edge_label = torch.cat((edge_label, edge_label))
    data = Data(edge_index=edge_index, edge_label=edge_label)
    return data


In [26]:
# NOTE: At training, we use interaction graph from train_df for train and validation
# NOTE: At inference, we can use graph of (train_df + valid_df)
train_graph = create_interaction_graph(train_df)
train_valid_graph = create_interaction_graph(pd.concat([train_df, valid_df], axis=0))

print(train_graph)
print(train_valid_graph)
print("---"*10)

print(train_graph.edge_index)

Data(edge_index=[2, 292256], edge_label=[292256])
Data(edge_index=[2, 348890], edge_label=[348890])
------------------------------
tensor([[   0,    0,    0,  ..., 8449, 8536, 8636],
        [  31,   98,  144,  ..., 2102, 2102, 2102]])


In [27]:
class UserItemPairDataset(Dataset):
    def __init__(self, df: pd.DataFrame):
        """
        Args:
            df (pd.DataFrame): with columns ['user_id', 'item_id', 'label']
        """
        self.user_ids = torch.tensor(df['userID'].values, dtype=torch.long)
        self.item_ids = torch.tensor(df['movieID'].values, dtype=torch.long)
        self.labels = torch.tensor(df['label'].values, dtype=torch.float)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx) -> Tuple[torch.tensor]:
        return self.user_ids[idx], self.item_ids[idx], self.labels[idx]

train_dataset = UserItemPairDataset(train_df)
valid_dataset = UserItemPairDataset(valid_df)
test_dataset = UserItemPairDataset(test_df)
print("train data count:", len(train_dataset))
print("valid data count:", len(valid_dataset))
print("test data count:", len(test_dataset))

train_loader = DataLoader(train_dataset, batch_size=512, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=512)
test_loader = DataLoader(test_dataset, batch_size=512)


train data count: 340207
valid data count: 64761
test data count: 75640


In [ ]:

# class GCNRecommender(LightningModule):
#     def __init__(
#             self, 
#             num_users: int, 
#             num_items: int, 
#             embedding_dim: int, 
#             edge_index: torch.tensor, # should be on device
#             lr=1e-3, 
#         ):
#         super().__init__()
#         self.save_hyperparameters()

#         self.user_embedding = nn.Embedding(num_users, embedding_dim)
#         self.item_embedding = nn.Embedding(num_items, embedding_dim)

#         nn.init.xavier_normal_(self.user_embedding.weight)
#         nn.init.xavier_normal_(self.item_embedding.weight)

#         self.edge_index = edge_index # [2, num_edges*2]
#         self.gcn = GCNConv(embedding_dim, embedding_dim)

#         self.lr = lr
#         self.criterion = nn.BCELoss()

#     def forward(self):
#         x = torch.cat([self.user_embedding.weight, self.item_embedding.weight], dim=0)  # [num_nodes, dim]
#         x = self.gcn(x, self.edge_index)
#         return x  # Updated embeddings of all users and items

#     def _shared_step(self, batch, stage):
#         x = self.forward()  # get full propagated embeddings

#         user_emb = x[batch['user']]
#         item_emb = x[batch['item'] + self.hparams.num_users]  # offset item ID in embedding table

#         logits = (user_emb * item_emb).sum(dim=1)
#         probs = torch.sigmoid(logits)
#         loss = self.criterion(probs, batch['label'])

#         preds = (probs > 0.5).int().cpu().numpy()
#         labels = batch['label'].int().cpu().numpy()

#         acc = accuracy_score(labels, preds)
#         prec = precision_score(labels, preds, zero_division=0)
#         rec = recall_score(labels, preds, zero_division=0)
#         f1 = f1_score(labels, preds, zero_division=0)

#         self.log(f"{stage}_loss", loss, on_step=False, on_epoch=True)
#         self.log(f"{stage}_acc", acc, on_step=False, on_epoch=True)
#         self.log(f"{stage}_precision", prec, on_step=False, on_epoch=True)
#         self.log(f"{stage}_recall", rec, on_step=False, on_epoch=True)
#         self.log(f"{stage}_f1", f1, on_step=False, on_epoch=True)

#         return loss

#     def training_step(self, batch, batch_idx):
#         return self._shared_step(batch, 'train')

#     def validation_step(self, batch, batch_idx):
#         return self._shared_step(batch, 'val')

#     def configure_optimizers(self):
#         return torch.optim.Adam(self.parameters(), lr=self.lr)


In [28]:
from common.mmgcn_v2 import GCN_ID

class GCNRecLightning(LightningModule):
    def __init__(self, edge_index, num_user, num_item, dim_id=64, lr=1e-3, device='cpu'):
        super().__init__()
        self.save_hyperparameters()

        self.model = GCN_ID(
            edge_index=edge_index,
            num_user=num_user,
            num_item=num_item,
            dim_id=dim_id,
            device=device
        )

        self.num_user = num_user
        self.num_item = num_item
        self.lr = lr

    def forward(self):
        return self.model()

    def compute_scores(self, emb, user_idx, item_idx):
        # TODO: why adding self.num_user?
        # return (emb[user_idx] * emb[item_idx + self.num_user]).sum(dim=1)
        return (emb[user_idx] * emb[item_idx]).sum(dim=1)

    def shared_step(self, batch):
        user, item, label = batch
        emb = self.forward()
        scores = self.compute_scores(emb, user, item)
        loss = F.binary_cross_entropy_with_logits(scores, label)

        preds = torch.sigmoid(scores) > 0.5
        acc = accuracy_score(label.cpu(), preds.cpu())
        prec = precision_score(label.cpu(), preds.cpu())
        rec = recall_score(label.cpu(), preds.cpu())
        f1 = f1_score(label.cpu(), preds.cpu())

        return loss, acc, prec, rec, f1

    def training_step(self, batch, batch_idx):
        loss, acc, prec, rec, f1 = self.shared_step(batch)
        self.log_dict({'train_loss': loss, 'train_acc': acc, 'train_prec': prec, 'train_rec': rec, 'train_f1': f1})
        return loss

    def validation_step(self, batch, batch_idx):
        loss, acc, prec, rec, f1 = self.shared_step(batch)
        self.log_dict({'val_loss': loss, 'val_acc': acc, 'val_prec': prec, 'val_rec': rec, 'val_f1': f1}, prog_bar=True)
        return loss

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.lr)

In [29]:
# setup MLflow logger and callbacks
from pytorch_lightning.loggers import MLFlowLogger
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping, Timer
RUN_NAME = "gcn-baseline-test"

mlflow_logger = MLFlowLogger(
    experiment_name="gcn-bce-exp",
    run_name=RUN_NAME,
    tracking_uri=MLFLOW_SERVICE_URI,  # can also use http://... for remote
)

checkpoint_callback = ModelCheckpoint(
    monitor="val_f1",  # or "val_f1"
    mode="max",           # or "max" if you're monitoring accuracy/F1
    save_top_k=1,
    save_weights_only=True,
    dirpath="test_checkpoints/",
    filename=f"{RUN_NAME}-best-checkpoint-{{epoch:02d}}-{{val_f1:.2f}}",
    verbose=True
)

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=3,
    mode="min",
    verbose=True
)

timer = Timer()


In [30]:
trainer = Trainer(
    max_epochs=10,
    logger=mlflow_logger,
    callbacks=[
        checkpoint_callback,
        early_stopping,
        timer,
    ],
    accelerator='cpu',  # or 'auto'
    # devices=1
)


GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/Users/jefferybai/Desktop/Master/BILAB/Master Thesis/DPRecSys/.venv/lib/python3.11/site-packages/pytorch_lightning/trainer/setup.py:177: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.


In [35]:
model = GCNRecLightning(
    edge_index=train_graph.edge_index,  # shape [2, num_edges]
    num_user=NUM_USER,
    num_item=NUM_ITEM,
    dim_id=64,
    lr=1e-3,
    device='cuda' if torch.cuda.is_available() else 'cpu'
)

trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=valid_loader)


Experiment with name gcn-bce-exp not found. Creating it.

  | Name  | Type   | Params | Mode 
-----------------------------------------
0 | model | GCN_ID | 49.5 K | train
-----------------------------------------
49.5 K    Trainable params
0         Non-trainable params
49.5 K    Total params
0.198     Total estimated model params size (MB)
13        Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/Users/jefferybai/Desktop/Master/BILAB/Master Thesis/DPRecSys/.venv/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:425: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.
/Users/jefferybai/Desktop/Master/BILAB/Master Thesis/DPRecSys/.venv/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:425: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


Training: |          | 0/? [00:00<?, ?it/s]


Detected KeyboardInterrupt, attempting graceful shutdown ...


NameError: name 'exit' is not defined